# Features and Overfitting: How a Model Memorizes, and How L1 and L2 Stop It

The runnable code of [Features and Overfitting: How a Model Memorizes, and How L1 and L2 Stop It](https://residual-stream.lvtan71.workers.dev/posts/features-overfitting-regularization/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/features-overfitting-regularization.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "features-overfitting-regularization"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/features-overfitting-regularization/overfit.py', 'shared/features-overfitting-regularization/router_data.py', 'data/router-questions.csv']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## A real dataset

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/features-overfitting-regularization/#a-real-dataset)

In [ ]:
%%writefile router_data.py
import csv
import re
import numpy as np

ROUTES = ["serving", "architecture", "agents", "foundations"]
rows = list(csv.DictReader(open("router-questions.csv")))
english = [r for r in rows if r["lang"] == "en"]
vietnamese = [r for r in rows if r["lang"] == "vi"]

def tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def split(seed):
    """30 train / 10 validation / 10 test questions per route."""
    rng = np.random.default_rng(seed)
    train, val, test = [], [], []
    for route in ROUTES:
        group = [r for r in english if r["route"] == route]
        for k, i in enumerate(rng.permutation(len(group))):
            (train if k < 30 else val if k < 40 else test).append(group[i])
    return train, val, test

def featurize(train, *others):
    """Bag of words: one 0/1 column per word seen in *training* questions."""
    vocab = sorted({t for r in train for t in tokens(r["question"])})
    column = {t: j for j, t in enumerate(vocab)}
    out = []
    for rs in (train, *others):
        X = np.zeros((len(rs), len(vocab)))
        for i, r in enumerate(rs):
            for t in set(tokens(r["question"])):
                if t in column:          # words never seen in training are dropped
                    X[i, column[t]] = 1
        y = np.array([r["route"] == "serving" for r in rs], dtype=float)
        out.append((X, y))
    return out, vocab

## Overfitting, watched

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/features-overfitting-regularization/#overfitting-watched)

In [ ]:
%%writefile overfit.py
import numpy as np
from router_data import split, featurize

sigmoid = lambda z: 1 / (1 + np.exp(-z))

def log_loss(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

def train(X, y, lam=0.0, penalty="l2", eta=0.5, steps=4000):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(steps):
        p = sigmoid(X @ w + b)
        grad_w, grad_b = X.T @ (p - y) / len(y), (p - y).mean()
        if penalty == "l2":                   # weight decay
            w -= eta * (grad_w + lam * w)
        else:                                 # L1: step, then soft-threshold
            w -= eta * grad_w
            w = np.sign(w) * np.maximum(np.abs(w) - eta * lam, 0.0)
        b -= eta * grad_b
    return w, b

if __name__ == "__main__":
    # 1. One split, no penalty: watch training and validation loss diverge.
    (train_set, val_set, _), vocab = featurize(*split(seed=0))
    (Xtr, ytr), (Xva, yva) = train_set, val_set
    print(f"{len(vocab)} words, {len(ytr)} training questions")
    w, b = np.zeros(Xtr.shape[1]), 0.0
    for step in range(1, 25601):
        p = sigmoid(Xtr @ w + b)
        w -= 0.5 * Xtr.T @ (p - ytr) / len(ytr)
        b -= 0.5 * (p - ytr).mean()
        if step in (100, 400, 1600, 3200, 6400, 25600):
            print(f"step {step:5d}: train loss {log_loss(sigmoid(Xtr @ w + b), ytr):.4f}"
                  f"  val loss {log_loss(sigmoid(Xva @ w + b), yva):.4f}"
                  f"  |w| {np.linalg.norm(w):5.2f}")

In [ ]:
%run overfit.py

The post printed:

```text
379 words, 120 training questions
step   100: train loss 0.2138  val loss 0.3951  |w|  4.02
step   400: train loss 0.0701  val loss 0.3196  |w|  8.32
step  1600: train loss 0.0175  val loss 0.2880  |w| 13.43
step  3200: train loss 0.0086  val loss 0.2870  |w| 16.01
step  6400: train loss 0.0043  val loss 0.2918  |w| 18.56
step 25600: train loss 0.0011  val loss 0.3117  |w| 23.64
```

## The sweep

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/features-overfitting-regularization/#the-sweep)

In [ ]:
import numpy as np
from router_data import split, featurize
from overfit import sigmoid, log_loss, train

# 2. Average every setting over 20 random splits: one 40-question
#    validation set is too small to trust on its own.
def evaluate(lam, penalty, seeds=range(20)):
    tr_acc, va_loss, va_acc, nonzero = [], [], [], []
    for seed in seeds:
        ((Xtr, ytr), (Xva, yva), _), _ = featurize(*split(seed))
        w, b = train(Xtr, ytr, lam, penalty)
        p = sigmoid(Xva @ w + b)
        tr_acc.append(((sigmoid(Xtr @ w + b) >= 0.5) == ytr).mean())
        va_loss.append(log_loss(p, yva))
        va_acc.append(((p >= 0.5) == yva).mean())
        nonzero.append((np.abs(w) > 0).sum())
    se = np.std(va_loss) / np.sqrt(len(va_loss))
    return (f"{penalty} λ={lam:<6}: train acc {np.mean(tr_acc):.3f}  "
            f"val loss {np.mean(va_loss):.3f} ± {se:.3f}  "
            f"val acc {np.mean(va_acc):.3f}  nonzero weights {np.mean(nonzero):.0f}")

for lam in [0.0, 0.001, 0.003, 0.01, 0.03, 0.1]:
    print(evaluate(lam, "l2"))
for lam in [0.001, 0.004, 0.008, 0.016]:
    print(evaluate(lam, "l1"))

The post printed:

```text
l2 λ=0.0   : train acc 1.000  val loss 0.480 ± 0.034  val acc 0.818  nonzero weights 383
l2 λ=0.001 : train acc 1.000  val loss 0.421 ± 0.021  val acc 0.804  nonzero weights 383
l2 λ=0.003 : train acc 1.000  val loss 0.416 ± 0.015  val acc 0.794  nonzero weights 383
l2 λ=0.01  : train acc 0.987  val loss 0.435 ± 0.010  val acc 0.784  nonzero weights 383
l2 λ=0.03  : train acc 0.887  val loss 0.473 ± 0.006  val acc 0.753  nonzero weights 383
l2 λ=0.1   : train acc 0.751  val loss 0.516 ± 0.002  val acc 0.750  nonzero weights 383
l1 λ=0.001 : train acc 1.000  val loss 0.530 ± 0.029  val acc 0.790  nonzero weights 112
l1 λ=0.004 : train acc 1.000  val loss 0.514 ± 0.022  val acc 0.767  nonzero weights 51
l1 λ=0.008 : train acc 0.893  val loss 0.502 ± 0.012  val acc 0.768  nonzero weights 27
l1 λ=0.016 : train acc 0.772  val loss 0.532 ± 0.005  val acc 0.742  nonzero weights 10
```

In [ ]:
import numpy as np
from router_data import split, featurize, tokens
from overfit import sigmoid, log_loss, train

# 3. Which words does L1 keep? (one split, λ = 0.008)
((Xtr, ytr), _, _), vocab = featurize(*split(seed=0))
w, b = train(Xtr, ytr, 0.008, "l1")
kept = sorted((w[j], vocab[j]) for j in np.flatnonzero(w))
print(f"{len(kept)} of {len(vocab)} weights are nonzero; those of size >= 0.01:")
print("toward serving:", ", ".join(f"{t} {v:+.2f}" for v, t in kept[::-1] if v >= 0.01))
print("away:          ", ", ".join(f"{t} {v:+.2f}" for v, t in kept if v <= -0.01))

The post printed:

```text
30 of 379 weights are nonzero; those of size >= 0.01:
toward serving: batching +1.70, to +1.60, token +1.34, prefix +1.26, model +1.16, memory +1.14, decode +0.91, my +0.81, tokens +0.80, draft +0.74, one +0.68, vllm +0.63, request +0.61, for +0.50, does +0.33, on +0.28, throughput +0.22, cache +0.22, time +0.12
away:           agent -1.80, need -0.87, what -0.74, attention -0.39, how -0.37, should -0.12, layer -0.02
```

## One good feature matches 379

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/features-overfitting-regularization/#one-good-feature-matches-379)

In [ ]:
# 4. The hand-picked keyword count of "Learning Is Optimization" vs the learned bag of words.
SERVING = {"kv", "cache", "caching", "batch", "batching", "latency",
           "throughput", "serve", "serving", "vllm", "sglang", "tensorrt",
           "gpu", "decode", "prefill", "paged", "quantization", "memory"}
def hits(rs, words=SERVING):
    return np.array([[sum(t in words for t in tokens(r["question"]))]
                     for r in rs], dtype=float)
scores = {"keyword count": [], "keyword count, seen words only": [],
          "bag of words": [], "both": []}
for seed in range(20):
    tr, va, te = split(seed)
    ((Xtr, ytr), (Xva, yva), _), vocab = featurize(tr, va, te)
    seen = SERVING & set(vocab)             # keywords that occur in training
    designs = {"keyword count": (hits(tr), hits(va), 0.0),
               "keyword count, seen words only": (hits(tr, seen), hits(va, seen), 0.0),
               "bag of words": (Xtr, Xva, 0.003),
               "both": (np.c_[Xtr, hits(tr)], np.c_[Xva, hits(va)], 0.003)}
    for name, (A, B, lam) in designs.items():
        w, b = train(A, ytr, lam)
        p = sigmoid(B @ w + b)
        scores[name].append((log_loss(p, yva), ((p >= 0.5) == yva).mean()))
for name, s in scores.items():
    s = np.array(s)
    print(f"{name:31s} val loss {s[:, 0].mean():.3f}  val acc {s[:, 1].mean():.3f}")

The post printed:

```text
keyword count                   val loss 0.381  val acc 0.878
keyword count, seen words only  val loss 0.436  val acc 0.851
bag of words                    val loss 0.416  val acc 0.794
both                            val loss 0.347  val acc 0.866
```

## How sure are we?

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/features-overfitting-regularization/#how-sure-are-we)

In [ ]:
import numpy as np
from router_data import split, featurize
from overfit import sigmoid, log_loss, train

# 6. How sure are we? Compare two settings on the same 20 splits (paired),
#    with the Nadeau-Bengio correction for splits that share questions.
def val_loss(lam, steps=4000):
    out = []
    for seed in range(20):
        ((Xtr, ytr), (Xva, yva), _), _ = featurize(*split(seed))
        w, b = train(Xtr, ytr, lam, steps=steps)
        out.append(log_loss(sigmoid(Xva @ w + b), yva))
    return np.array(out)

def compare(a, b, label):
    d = a - b
    naive = d.std(ddof=1) / np.sqrt(len(d))
    honest = d.std(ddof=1) * np.sqrt(1 / len(d) + 40 / 120)   # n_val / n_train
    print(f"{label:37s} {d.mean():+.3f}   naive SE {naive:.3f}   corrected SE {honest:.3f}")

l2 = val_loss(0.003)
compare(l2, val_loss(0.0), "L2 0.003 - no penalty (4,000 steps)")
compare(l2, val_loss(0.0, steps=16000), "L2 0.003 - no penalty (16,000 steps)")
compare(l2, val_loss(0.001), "L2 0.003 - L2 0.001")
compare(l2, val_loss(0.01), "L2 0.003 - L2 0.01")

The post printed:

```text
L2 0.003 - no penalty (4,000 steps)   -0.064   naive SE 0.020   corrected SE 0.055
L2 0.003 - no penalty (16,000 steps)  -0.131   naive SE 0.030   corrected SE 0.082
L2 0.003 - L2 0.001                   -0.005   naive SE 0.006   corrected SE 0.017
L2 0.003 - L2 0.01                    -0.019   naive SE 0.006   corrected SE 0.016
```

## Don't let the test set make decisions

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/features-overfitting-regularization/#dont-let-the-test-set-make-decisions)

In [ ]:
%%writefile peek.py
import numpy as np
from router_data import split, featurize
from overfit import sigmoid, log_loss, train

# 5. Choose λ by the test set instead of the validation set, using the SAME
#    criterion (log-loss) on both sides, and report test log-loss.
LAMBDAS = [0.0, 0.001, 0.003, 0.01, 0.03, 0.1]
peek = []
for seed in range(20):
    ((Xtr, ytr), (Xva, yva), (Xte, yte)), _ = featurize(*split(seed))
    fits = [train(Xtr, ytr, lam) for lam in LAMBDAS]
    test_loss = [log_loss(sigmoid(Xte @ w + b), yte) for w, b in fits]
    val_loss = [log_loss(sigmoid(Xva @ w + b), yva) for w, b in fits]
    peek.append((min(test_loss), test_loss[int(np.argmin(val_loss))]))
peek = np.array(peek)
print(f"test loss, λ chosen on test: {peek[:, 0].mean():.3f};"
      f" chosen on validation: {peek[:, 1].mean():.3f}")

In [ ]:
%run peek.py

The post printed:

```text
test loss, λ chosen on test: 0.403; chosen on validation: 0.439
```